# Watch Agent

Pipeline de surveillance vidéo temps réel :
```
RTSP Stream
    → FrameSampler       (sous-échantillonnage temporel)
    → OnlineSceneSegmenter  (détection de scènes par histogramme HSV)
    → SceneDescriber     (description VLM via Qwen2.5VL:3b / Ollama)
```

## 1. Configuration

In [1]:
from pathlib import Path
from watchagent.settings import (
    OLLAMA_BASE_URL,
    OLLAMA_MODEL,
    DEFAULT_SOURCE_FPS,
    SAMPLER_TARGET_FPS,
    DESCRIBER_CONTEXT_WINDOW,
)

# --- Source vidéo ---
RTSP_URL = "rtsp://localhost:8554/live.stream"

# --- Sortie ---
OUTPUT_DIR = Path("scene_keyframes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ollama  : {OLLAMA_BASE_URL}  |  modèle : {OLLAMA_MODEL}")
print(f"FPS     : {DEFAULT_SOURCE_FPS} source  →  {SAMPLER_TARGET_FPS} analysés")
print(f"Contexte: {DESCRIBER_CONTEXT_WINDOW} scènes précédentes injectées")

Ollama  : http://localhost:11434  |  modèle : qwen2.5vl:3b
FPS     : 30.0 source  →  5.0 analysés
Contexte: 3 scènes précédentes injectées


## 2. Instanciation des composants

In [5]:
from watchagent.video_streaming import RTSPFrameStreamer
from watchagent.perception import FrameSampler, OnlineSceneSegmenter
from watchagent.vlm import OllamaVLMClient, SceneDescriber
from watchagent.agent import WatchAgent

# Tous les paramètres par défaut viennent de watchagent/settings.py
streamer    = RTSPFrameStreamer(rtsp_url=RTSP_URL)
sampler     = FrameSampler()
segmenter   = OnlineSceneSegmenter(frame_similarity_threshold=0.035)
vlm_client  = OllamaVLMClient()
describer   = SceneDescriber(client=vlm_client)

## 3. Callback : sauvegarde des keyframes et affichage des descriptions

In [6]:
import cv2

scene_log: list[dict] = []  # historique pour consultation post-run

def on_scene_described(scene, description: str) -> None:
    """Appelé après chaque description VLM (dans un thread VLM)."""

    # Sauvegarde des keyframes sur disque
    scene_dir = OUTPUT_DIR / f"scene_{scene['scene_index']:04d}"
    scene_dir.mkdir(parents=True, exist_ok=True)

    for i, (frame_idx, frame) in enumerate(scene["selected_keyframes"]):
        path = scene_dir / f"keyframe_{i:03d}_frame_{frame_idx}.jpg"
        cv2.imwrite(str(path), frame)

    # Sauvegarde description texte
    (scene_dir / "description.txt").write_text(
        f"scene_index : {scene['scene_index']}\n"
        f"status      : {scene['status']}\n"
        f"frames      : {scene['frames_count']}\n"
        f"start       : {scene['start_timestamp_utc']}\n"
        f"end         : {scene['end_timestamp_utc']}\n"
        f"\n{description}\n",
        encoding="utf-8",
    )

    scene_log.append({"scene": scene, "description": description})

## 4. Lancement de l'agent

Appuie sur **■ Stop** (Jupyter) ou **Ctrl+C** pour arrêter proprement.
Le WatchAgent attend la fin des descriptions VLM en cours avant de s'arrêter.

In [7]:

agent = WatchAgent(
    streamer=streamer,
    sampler=sampler,
    segmenter=segmenter,
    describer=describer,
    on_scene_described=on_scene_described,
    vlm_workers=1,   # 1 thread VLM suffit pour un modèle local
)

agent.run()

[rtsp @ 0x2f632dc0] method DESCRIBE failed: 404Not Found
[ WARN:0@498.285] global cap.cpp:215 open VIDEOIO(FFMPEG): backend is generally available but can't be used to capture by name
[rtsp @ 0x7871780674c0] method DESCRIBE failed: 404Not Found
[ WARN:2@499.291] global cap.cpp:215 open VIDEOIO(FFMPEG): backend is generally available but can't be used to capture by name


[STREAM] STREAMING


[rtsp @ 0x7871780674c0] method DESCRIBE failed: 404Not Found
[ WARN:2@591.554] global cap.cpp:215 open VIDEOIO(FFMPEG): backend is generally available but can't be used to capture by name
[rtsp @ 0x7871780674c0] method DESCRIBE failed: 404Not Found
[ WARN:2@592.560] global cap.cpp:215 open VIDEOIO(FFMPEG): backend is generally available but can't be used to capture by name
[rtsp @ 0x7871780674c0] method DESCRIBE failed: 404Not Found
[ WARN:2@593.562] global cap.cpp:215 open VIDEOIO(FFMPEG): backend is generally available but can't be used to capture by name
[rtsp @ 0x7871780674c0] method DESCRIBE failed: 404Not Found
[ WARN:2@594.564] global cap.cpp:215 open VIDEOIO(FFMPEG): backend is generally available but can't be used to capture by name
[rtsp @ 0x7871780674c0] method DESCRIBE failed: 404Not Found
[ WARN:2@595.567] global cap.cpp:215 open VIDEOIO(FFMPEG): backend is generally available but can't be used to capture by name
[rtsp @ 0x7871780674c0] method DESCRIBE failed: 404Not Found

[SCENE] index=0000  status=CLOSED_BY_DISCONNECT  frames=415  keyframes=38
[VLM] scene=0000  kf=38  chunks=10×4  res=320×240→320×240  dur=82.6s  ctx=0
[AGENT] Attente de 1 description(s) VLM en cours...
[VLM] scene=0000  chunk=1/10  85.2s
[VLM] scene=0000  chunk=2/10  81.4s
[VLM] scene=0000  chunk=3/10  77.0s
[VLM] scene=0000  chunk=4/10  78.6s
[VLM] scene=0000  chunk=5/10  79.8s
[VLM] scene=0000  chunk=6/10  78.5s
[VLM] scene=0000  chunk=7/10  78.5s
[VLM] scene=0000  chunk=8/10  79.5s
[VLM] scene=0000  chunk=9/10  80.4s
[VLM] scene=0000  chunk=10/10  79.3s

[VLM] scene_index=0000  (2026-04-12T19:18:14.948798+00:00 → 2026-04-12T19:19:48.250603+00:00)
**Contexte** : 
- **Type de lieu** : Salle de réunion ou de conférence, avec des murs blancs, des fenêtres à l'arrière, et une table centrale.
- **Conditions d'éclairage** : Lumière naturelle provenant des fenêtres et éclairage artificiel.

**Présence** :
- **Personnes** : Une seule personne, une femme, assise sur une chaise à côté de la ta

## 5. Consultation des résultats

In [10]:
for entry in scene_log:
    sc = entry["scene"]
    print(f"\n{'═'*60}")
    print(f" Scène {sc['scene_index']:04d}  |  {sc['status']}  |  {sc['frames_count']} frames")
    print(f" {sc['start_timestamp_utc']} → {sc['end_timestamp_utc']}")
    print(f"{'─'*60}")
    print(entry["description"])